# Seattle Building Permits — Exploratory Data Analysis

This notebook pulls the Seattle Building Permits dataset from the Seattle Open Data portal and explores it with a focus on accessory dwelling units (ADUs). The goal is to understand what people are actually building in Seattle backyards — which is often clearer than reading the municipal code.

**Dataset:** [Seattle Building Permits](https://data.seattle.gov/Permitting/Building-Permits/76t5-zqzr)  
**Source:** Seattle Open Data portal (Socrata API)  
**Output:** `data/raw/seattle_building_permits.csv`

## 1. Setup

In [1]:
import os
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

RAW_DATA_PATH = Path("../data/raw/seattle_building_permits.csv")
SOCRATA_TOKEN = os.getenv("SOCRATA_APP_TOKEN")
DATASET_ID = "76t5-zqzr"
BASE_URL = f"https://data.seattle.gov/resource/{DATASET_ID}.json"

## 2. Download dataset

Pull the full dataset from the Socrata API using pagination (500 rows per request). Skips the download if the file already exists locally.

In [2]:
def fetch_all_records(base_url: str, token: str | None, limit: int = 50000) -> list[dict]:
    """Page through the Socrata API and return all records."""
    headers = {"X-App-Token": token} if token else {}
    records = []
    offset = 0

    while True:
        resp = requests.get(
            base_url,
            headers=headers,
            params={"$limit": limit, "$offset": offset},
        )
        resp.raise_for_status()
        batch = resp.json()
        records.extend(batch)
        print(f"  fetched {len(records):,} records...", end="\r")
        if len(batch) < limit:
            break
        offset += limit

    print(f"\nDone — {len(records):,} total records")
    return records


if RAW_DATA_PATH.exists():
    print(f"File already exists at {RAW_DATA_PATH} — skipping download")
else:
    print("Fetching Seattle Building Permits...")
    records = fetch_all_records(BASE_URL, SOCRATA_TOKEN)
    df_raw = pd.DataFrame(records)
    RAW_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_raw.to_csv(RAW_DATA_PATH, index=False)
    print(f"Saved to {RAW_DATA_PATH}")

Fetching Seattle Building Permits...
  fetched 188,921 records...
Done — 188,921 total records
Saved to ../data/raw/seattle_building_permits.csv


In [3]:
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

Loaded 188,921 rows × 40 columns


## 3. Schema inspection

Profile every column: type, null rate, unique value count, and sample values. Anything surprising gets flagged — these observations feed directly into the dbt staging model and test design.

In [4]:
# Column-level profile: type, null rate, unique count, sample values
profile = pd.DataFrame({
    "dtype": df.dtypes,
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(1),
    "unique_count": df.nunique(),
    "sample_values": [df[c].dropna().unique()[:3].tolist() for c in df.columns],
})
profile

,dtype,null_count,null_pct,unique_count,sample_values
permitnum,str,0,0.0,188921,"[3003045-EX, 3003088-EX, 3003261-EX]"
permitclass,str,6869,3.6,6,"[Multifamily, Single Family/Duplex, Commercial]"
permitclassmapped,str,6869,3.6,2,"[Residential, Non-Residential]"
permittypemapped,str,0,0.0,5,[ECA and Shoreline Exemption/Street Improvemen...
permittypedesc,str,12586,6.7,18,"[Environmentally Critical Area Exemption, Shor..."
description,str,723,0.4,156714,[Exception/Exemption Request for: Land Use Per...
housingunits,float64,0,0.0,389,"[0.0, -6.0, 4.0]"
statuscurrent,str,0,0.0,24,"[Completed, Canceled, Corrections Required]"
relatedmup,str,174587,92.4,7815,"[3003045-LU, 3003088-LU, 3003261-LU]"
originaladdress1,str,907,0.5,87620,"[4739 35TH AVE S, 820 28TH AVE S, 3018 EAST LA..."


In [5]:
# Columns with high null rates (>50%) — flag as unreliable for dbt tests
high_null = profile[profile["null_pct"] > 50].sort_values("null_pct", ascending=False)
print(f"{len(high_null)} columns with >50% nulls:")
high_null[["dtype", "null_pct"]]

15 columns with >50% nulls:


,dtype,null_pct
relatedmup,str,92.4
daysissuepermitcity,float64,88.0
contractorcompanyname,str,84.2
totaldaysplanreview,float64,83.7
daysplanreviewcity,float64,83.7
planreviewcompletedate,str,83.6
daysinitialplanreview,float64,83.4
initialreviewcompletedate,str,83.4
parentpermitnum,str,79.5
readytoissuedate,str,72.7


In [6]:
# Date columns — check parsing and range
date_cols = [c for c in df.columns if "date" in c.lower()]
for col in date_cols:
    parsed = pd.to_datetime(df[col], errors="coerce")
    valid = parsed.dropna()
    print(f"{col}: {valid.min().date()} → {valid.max().date()} ({parsed.isnull().sum():,} unparseable)")

applieddate: 1986-04-28 → 2026-04-01 (44,679 unparseable)
issueddate: 1986-07-03 → 2026-04-01 (51,686 unparseable)
expiresdate: 2001-10-24 → 2028-03-20 (51,547 unparseable)
completeddate: 2005-02-22 → 2026-04-01 (84,146 unparseable)
readytoissuedate: 2017-10-11 → 2026-04-01 (137,405 unparseable)
initialreviewcompletedate: 2018-04-30 → 2026-04-01 (157,585 unparseable)
planreviewcompletedate: 2018-04-30 → 2026-04-01 (157,979 unparseable)


In [7]:
# Permit type distribution
df["permittypemapped"].value_counts().head(20)

Using column: permittypemapped


permittypemapped
Building                                                            151753
ECA and Shoreline Exemption/Street Improvement Exception Request     18515
Demolition                                                           15779
Roof                                                                  1823
Grading                                                               1051
Name: count, dtype: int64

## 4. Identify ADU-relevant permit types and codes

Search for DADU, ADU, backyard cottage, and related terms across permit type, description, and category fields. The goal is to isolate the exact filters needed to define an "ADU permit" for all downstream analysis.

In [8]:
# Terms to search across all string columns
ADU_TERMS = ["adu", "dadu", "accessory dwelling", "backyard cottage", "in-law", "carriage house"]

# Identify string columns to search
str_cols = df.select_dtypes(include="object").columns.tolist()

# Search each column for any ADU term (case-insensitive)
hits = {}
for col in str_cols:
    mask = df[col].str.lower().str.contains("|".join(ADU_TERMS), na=False)
    if mask.any():
        hits[col] = df.loc[mask, col].value_counts().head(10)
        print(f"✓ '{col}' — {mask.sum():,} matches")

print(f"\nTotal columns with ADU-related content: {len(hits)}")

/var/folders/sl/sgsbkmg917x7ythy3b64mx_80000gn/T/ipykernel_16556/568696317.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns.tolist()


✓ 'description' — 12,592 matches
✓ 'contractorcompanyname' — 34 matches
✓ 'housingcategory' — 369 matches
✓ 'dwellingunittype' — 6,432 matches

Total columns with ADU-related content: 4


In [9]:
# Show top matching values per column
for col, counts in hits.items():
    print(f"\n--- {col} ---")
    print(counts.to_string())


--- description ---
description
Allow new detached accessory dwelling unit to existing single family use per land use code. Construct new one family dwelling  per plan.                   46
Construct a detached accessory dwelling unit  per plans                                                                                                    44
Construct detached accessory dwelling unit (DADU) to existing single family residence  per plan.                                                           39
Construct detached accessory dwelling unit (DADU) to existing single-family residence  per plan                                                            34
Construct detached accessory dwelling unit [DADU]  per plan.                                                                                               33
Allow new detached accessory dwelling unit to existing single family use per land use code.  Construct new one family dwelling  per plan.                  28
Establish use and C

In [10]:
# Build the ADU filter — update this based on what the search reveals above
# This is the canonical definition used in all downstream analysis
adu_mask = df[str_cols].apply(
    lambda col: col.str.lower().str.contains("|".join(ADU_TERMS), na=False)
).any(axis=1)

df_adu = df[adu_mask].copy()
print(f"ADU permits identified: {len(df_adu):,} ({len(df_adu)/len(df)*100:.1f}% of all permits)")

ADU permits identified: 12,869 (6.8% of all permits)


## 5. Volume and trend analysis

Basic charts on the ADU permit subset: applications by year, by neighbourhood, approval timelines, and value of work distributions. These become the core visuals for Substack Post 1.

In [11]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

# Parse application date
app_date_col = "applieddate"
df_adu["application_date"] = pd.to_datetime(df_adu[app_date_col], errors="coerce")
df_adu["year"] = df_adu["application_date"].dt.year

print(f"Application date column: {app_date_col}")

KeyError: None

In [12]:
 df_adu.columns.tolist()

['permitnum',
 'permitclass',
 'permitclassmapped',
 'permittypemapped',
 'permittypedesc',
 'description',
 'housingunits',
 'statuscurrent',
 'relatedmup',
 'originaladdress1',
 'originalcity',
 'originalstate',
 'originalzip',
 'contractorcompanyname',
 'link',
 'latitude',
 'longitude',
 'location1',
 'daysoutcorrections',
 'numberreviewcycles',
 'dependentbuilding',
 'housingcategory',
 'estprojectcost',
 'applieddate',
 'issueddate',
 'expiresdate',
 'completeddate',
 'housingunitsremoved',
 'housingunitsadded',
 'zoning',
 'standardplan',
 'dwellingunittype',
 'parentpermitnum',
 'readytoissuedate',
 'totaldaysplanreview',
 'daysinitialplanreview',
 'daysplanreviewcity',
 'initialreviewcompletedate',
 'planreviewcompletedate',
 'daysissuepermitcity']

In [ ]:
# ADU permits by year
yearly = df_adu.groupby("year").size().reset_index(name="count")
yearly = yearly[yearly["year"] >= 2000]  # filter out data quality outliers

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(yearly["year"], yearly["count"], color=sns.color_palette("muted")[0])
ax.set_title("Seattle ADU Permit Applications by Year", fontsize=14)
ax.set_xlabel("Year")
ax.set_ylabel("Number of permits")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

In [ ]:
# Top zip codes by ADU permit volume (no neighbourhood column in dataset)
zip_counts = df_adu["originalzip"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 6))
zip_counts.plot(kind="barh", ax=ax, color=sns.color_palette("muted")[1])
ax.set_title("Top 15 Zip Codes by ADU Permit Volume", fontsize=14)
ax.set_xlabel("Number of permits")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Approval timeline: days from application to issue
df_adu["issue_date"] = pd.to_datetime(df_adu["issueddate"], errors="coerce")
df_adu["days_to_approval"] = (df_adu["issue_date"] - df_adu["application_date"]).dt.days
valid_timelines = df_adu["days_to_approval"].dropna()
valid_timelines = valid_timelines[(valid_timelines >= 0) & (valid_timelines < 1500)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(valid_timelines, bins=60, color=sns.color_palette("muted")[2], edgecolor="white")
ax.axvline(valid_timelines.median(), color="red", linestyle="--", label=f"Median: {valid_timelines.median():.0f} days")
ax.set_title("ADU Permit Approval Timeline (Application → Issue)", fontsize=14)
ax.set_xlabel("Days")
ax.set_ylabel("Number of permits")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Median: {valid_timelines.median():.0f} days | Mean: {valid_timelines.mean():.0f} days | 90th pct: {valid_timelines.quantile(0.9):.0f} days")

In [ ]:
# Value of work distribution
df_adu["value_clean"] = pd.to_numeric(df_adu["estprojectcost"], errors="coerce")
values = df_adu["value_clean"].dropna()
values = values[(values > 0) & (values < values.quantile(0.99))]  # trim top 1% outliers

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(values / 1000, bins=60, color=sns.color_palette("muted")[3], edgecolor="white")
ax.axvline(values.median() / 1000, color="red", linestyle="--", label=f"Median: ${values.median()/1000:.0f}k")
ax.set_title("ADU Permit — Estimated Project Cost", fontsize=14)
ax.set_xlabel("Cost ($thousands)")
ax.set_ylabel("Number of permits")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Median: ${values.median():,.0f} | Mean: ${values.mean():,.0f} | 90th pct: ${values.quantile(0.9):,.0f}")

## 6. Summary of findings and data quality notes

Document what was found and flag issues for the dbt layer. Update this section after running the notebook.

In [ ]:
# Data quality issues to carry into dbt tests
# Update this after running the full notebook — these become the basis for dbt tests

data_quality_notes = {
    "high_null_columns": "See section 3 — columns with >50% nulls should not be used in NOT NULL tests",
    "date_parsing": "Some date fields contain unparseable values — cast carefully in staging",
    "value_outliers": "Estimated value field has extreme high-end outliers — consider capping or flagging",
    "adu_filter": f"ADU search terms used: {ADU_TERMS} — review matches manually to confirm precision",
    "year_filter": "Pre-2000 records appear to have data quality issues — filtered from trend analysis",
}

print("Data quality notes for dbt layer:")
for k, v in data_quality_notes.items():
    print(f"\n  [{k}]\n  {v}")